# OCP7 - Réalisez une analyse de sentiments grâce au Deep Learning
# Notebook 4 - V3 du Script python pour une classification de tweets - modèle avancé avec Transformers

In [1]:
# Example code to display the version of all imports
import pkg_resources

# List of package names
packages = ['numpy',
           'torch',
           'transformers']

for package in packages:
    version = pkg_resources.get_distribution(package).version
    print(f"{package} == {version}")
    print()

numpy == 1.23.5

torch == 2.1.2

transformers == 4.36.2



In [2]:
'''
This script uses the following python and packages versions :

python == 3.10.9 | packaged by Anaconda.

numpy == 1.23.5

torch == 2.1.2

transformers == 4.36.2

'''

import numpy as np
import torch
from transformers import BertModel, BertTokenizer
from torch import nn

# Set GPU
device = torch.device("cpu")

# Define class names, to denote a negative or positive sentiment
class_names = ['negative', 'positive']

# Set the model name
MODEL_NAME = 'bert-base-cased'

# Build a BERT based tokenizer
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# Set max len of tweets as 150 tokens
MAX_LEN = 150

# Build the Sentiment Classifier class
class SentimentClassifier(nn.Module):
    def __init__(self, n_classes):
        super(SentimentClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        bert_output = self.bert(
          input_ids=input_ids,
          attention_mask=attention_mask
        )
        pooled_output = bert_output.pooler_output
        output = self.drop(pooled_output)
        return self.out(output)

# Instantiate the model and move to classifier
model = SentimentClassifier(len(class_names))
model = model.to(device)

# Load the best state of the model
model_path = 'best_model_state.bin'
model.load_state_dict(torch.load(model_path, map_location='cpu'))

# Define a new text
review_text = "This plan is giving me headaches."

# Encode the text
encoded_review = tokenizer.encode_plus(
    review_text,
    max_length=MAX_LEN,
    add_special_tokens=True,
    return_token_type_ids=False,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt',
)

# Set input_ids and attention_mask
input_ids = encoded_review['input_ids'].to(device)
attention_mask = encoded_review['attention_mask'].to(device)

# Get the output
output = model(input_ids, attention_mask)
_, prediction = torch.max(output, dim=1)

# Display the text and associated sentiment prediction
print(f'Tweet: {review_text}')
print(f'Sentiment  : {class_names[prediction]}')

Tweet: This plan is giving me headaches.
Sentiment  : negative
